In [1]:
import pennylane as qml
from pennylane import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

In [4]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [5]:
pca = PCA(n_components=4)

X = pca.fit_transform(X)

In [6]:
X = X / np.max(np.abs(X)) * np.pi

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
n_qubits = 4

dev = qml.device("default.qubit", wires=n_qubits)

In [9]:
@qml.qnode(dev)

def circuit(inputs, weights):

    qml.AngleEmbedding(inputs, wires=range(n_qubits))

    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))

    return qml.expval(qml.PauliZ(0))

In [10]:
n_layers = 2

weights = np.random.randn(
    n_layers,
    n_qubits,
    3,
    requires_grad=True
)

In [11]:
def sigmoid(x):
    return 1/(1+np.exp(-x))

In [12]:
def predict_proba(x, weights):

    out = circuit(x, weights)

    return sigmoid(out)

In [13]:
def loss(weights):

    predictions = np.array(
        [predict_proba(x, weights) for x in X_train]
    )

    eps = 1e-8

    return -np.mean(
        y_train*np.log(predictions+eps)
        +(1-y_train)*np.log(1-predictions+eps)
    )

In [14]:
optimizer = qml.AdamOptimizer(stepsize=0.05)

In [15]:
epochs = 50

for epoch in range(epochs):

    weights = optimizer.step(loss, weights)

    if epoch % 5 == 0:
        print(epoch, loss(weights))

0 0.7663147532891673
5 0.6850736043530018
10 0.6470029470343731
15 0.6383159691077946
20 0.6382554000997981
25 0.6367454754770768
30 0.6339659256455948
35 0.6298953682135106
40 0.6244324879303446
45 0.6171012232741471


In [16]:
probs = np.array([
    predict_proba(x, weights)
    for x in X_test
])

predictions = (probs > 0.5).astype(int)

In [17]:
print("Accuracy :", accuracy_score(y_test, predictions))

print(classification_report(y_test, predictions))

Accuracy : 0.6578947368421053
              precision    recall  f1-score   support

           0       1.00      0.09      0.17        43
           0       1.00      0.09      0.17        43
           1       0.65      1.00      0.78        71
           1       0.65      1.00      0.78        71

    accuracy                           0.66       228
   macro avg       0.82      0.55      0.48       228
weighted avg       0.78      0.66      0.55       228

